# Can we predict 12-month loan default?
Target: `defaulted` (1 = defaulted within 12 months of funding). Plan: inspect data quality, explore, then compare a baseline against a few models with cross-validation, and evaluate once on a held-out test set. Metrics focus on ROC AUC and PR AUC because defaults are rare (~10%), so accuracy is misleading.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
SEED=0
df=pd.read_csv('data.csv'); dd=pd.read_csv('columns.csv')
dd

## 1. Data quality

In [ ]:
print(df.shape)
print('missing values:\n', df.isna().sum().to_string())
print('duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
df.describe().T

In [ ]:
rate=df.defaulted.mean(); print(f'Default rate: {rate:.3%} ({df.defaulted.sum()} of {len(df)})')
# sanity checks on ranges
print((df[['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']]<0).sum().to_string())
print('employment_years > age-14:', (df.employment_years>df.age-14).sum())
print('employment_years == 0:', (df.employment_years==0).sum())

## 2. Exploration
Default rate by feature decile/category, to see which signals are present.

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    b=pd.qcut(df[c],10,duplicates='drop')
    g=df.groupby(b,observed=True).defaulted.mean()
    a.plot(range(len(g)),g.values,marker='o'); a.axhline(rate,ls='--',c='grey')
    a.set_title(c); a.set_xlabel('decile (low→high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','count']))

In [ ]:
df[num+['defaulted']].corr()['defaulted'].drop('defaulted').sort_values()

## 3. Modelling setup
`application_id` is an identifier, so it is dropped. There is no date column, so I can't do a time-based split; a stratified random split is used (a limitation: it can't tell us how the model holds up under drift). 20% is held out and untouched until the final evaluation; model comparison uses 5-fold CV on the training part.

In [ ]:
X=df.drop(columns=['defaulted','application_id']); y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
cat=['region']
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(drop='first'),cat)])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'gradient boosting':make_pipeline(ColumnTransformer([('num','passthrough',num),('cat',OneHotEncoder(),cat)]),
        HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED)),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[];oof={}
for n,m in models.items():
    p=cross_val_predict(m,Xtr,ytr,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    rows.append(dict(model=n,cv_auc=roc_auc_score(ytr,p),cv_pr_auc=average_precision_score(ytr,p),cv_brier=brier_score_loss(ytr,p)))
pd.DataFrame(rows).set_index('model').round(4)

## 4. Held-out test evaluation

In [ ]:
res={}
for n,m in models.items():
    m.fit(Xtr,ytr); res[n]=m.predict_proba(Xte)[:,1]
tab=pd.DataFrame({n:dict(auc=roc_auc_score(yte,p),pr_auc=average_precision_score(yte,p),brier=brier_score_loss(yte,p)) for n,p in res.items()}).T
display(tab.round(4))
# bootstrap CI for AUC
rng=np.random.default_rng(SEED)
def boot(p,n=1000):
    idx=np.arange(len(yte)); out=[]
    yv=yte.values
    for _ in range(n):
        i=rng.choice(idx,len(idx))
        out.append(roc_auc_score(yv[i],p[i]))
    return np.percentile(out,[2.5,97.5])
for n in ['logistic','gradient boosting']: print(n,'test AUC 95% CI:',boot(res[n]).round(3))
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n,p in res.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC')
ax[1].set(xlabel='recall',ylabel='precision',title='Precision-recall'); ax[1].legend(); plt.show()

### Practical value: ranking and calibration

In [ ]:
p=res['logistic']; t=pd.DataFrame({'p':p,'y':yte.values})
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
g=t.groupby('decile',observed=True).agg(n=('y','size'),pred=('p','mean'),actual=('y','mean'))
display(g.round(3))
top=t.nlargest(len(t)//5,'p').y.sum()/t.y.sum()
print(f'Top 20% riskiest applications capture {top:.0%} of all defaults (random would capture 20%).')
fig,a=plt.subplots(figsize=(4.5,4.5)); fp,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile')
a.plot(mp,fp,marker='o'); a.plot([0,.4],[0,.4],'k:'); a.set(xlabel='predicted',ylabel='observed',title='Calibration (logistic)'); plt.show()

## 5. What drives the prediction?

In [ ]:
m=models['logistic']; m.fit(Xtr,ytr)
pi=permutation_importance(m,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,index=X.columns); err=pd.Series(pi.importances_std,index=X.columns)
imp.sort_values().plot.barh(xerr=err[imp.sort_values().index]); plt.xlabel('drop in test AUC when shuffled'); plt.show()
coef=pd.Series(m[-1].coef_[0],index=m[0].get_feature_names_out()); coef.sort_values()

## 6. Threshold trade-off

In [ ]:
yv=yte.values
rows=[]
for th in [0.05,0.08,0.10,0.15,0.20,0.30]:
    pred=p>=th; tp=(pred&(yv==1)).sum()
    rows.append(dict(threshold=th,flagged=pred.mean(),precision=tp/max(pred.sum(),1),recall=tp/yv.sum()))
pd.DataFrame(rows).round(3)

## Conclusions
**Yes, defaults are predictable, but only moderately.**

- Base rate is 10.5% (837 / 8,000). A model using the six numeric features plus region reaches a **held-out ROC AUC of about 0.78** (95% bootstrap CI 0.745–0.813; 5-fold CV gave 0.775) and a PR AUC of about 0.31, against 0.10 for the base rate.
- Plain **logistic regression did as well as or better than gradient boosting** (test AUC 0.781 vs 0.762, with overlapping CIs). The signal looks roughly additive, so there is no case for the more complex model here.
- The model **ranks well and its probabilities are well calibrated**. The riskiest decile defaults at about 35% against about 0.6% in the safest, and the top 20% of applications capture about 53% of all defaults.
- Drivers: higher `debt_ratio` and `loan_amount_k` raise risk, and higher `income_k` and `employment_years` lower it. `age`, `credit_lines` and `region` add essentially nothing.
- **Precision is limited.** Flagging the top ~14% (threshold 0.20) catches 44% of defaults at 33% precision, and catching 70% of defaults means flagging a third of applicants at 22% precision. The right threshold depends on the cost of a missed default versus a rejected good loan, which is not in this data.

**Caveats**
- Stratified random split with no date column, so I could not test stability over time. Check this before relying on the model.
- 434 rows have `employment_years > age - 14`, and 720 rows have exactly 0 years. These may be data-entry artefacts or a default code, and are worth checking with the data owner.
- Only 837 positives, so the estimates are noisy, as the CI widths show. Hyperparameters were not heavily tuned.
- Association only, not causal. Using this for real lending decisions would also need a fairness and compliance review.
